In [2]:
import torch
import torch.nn
import torch.optim
import torchvision
import torchvision.transforms
import time

transform_train = torchvision.transforms.Compose([
    torchvision.transforms.RandomCrop(32, padding=4),
    torchvision.transforms.RandomHorizontalFlip(),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

transform_test = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

train_set = torchvision.datasets.CIFAR10(root='./cifar10_data', train=True, download=True, transform=transform_train)
test_set = torchvision.datasets.CIFAR10(root='./cifar10_data', train=False, download=True, transform=transform_test)

train_loader = torch.utils.data.DataLoader(train_set, batch_size=128, shuffle=True, num_workers=2)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=128, shuffle=False, num_workers=2)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

model = torchvision.models.resnet18(weights=None, num_classes=10)
model = model.to(device)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4)

for epoch in range(1, 101):
    start_time = time.time()

    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        train_total += labels.size(0)
        train_correct += predicted.eq(labels).sum().item()

    epoch_time = time.time() - start_time
    train_loss = train_loss / train_total
    train_acc = 100.0 * train_correct / train_total

    print(f"Epoch {epoch:2d}: TrainTime = {epoch_time:5.2f}s, "
          f"TrainLoss = {train_loss:.4f}, TrainAcc = {train_acc:.2f}%", end=' ')

    if train_acc >= 90.0:
        model.eval()
        test_correct = 0
        test_total = 0
        with torch.no_grad():
            for inputs, labels in test_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, predicted = outputs.max(1)
                test_total += labels.size(0)
                test_correct += predicted.eq(labels).sum().item()
        test_acc = 100.0 * test_correct / test_total
        print(f"TestAcc = {test_acc:.2f}%")
    else:
        print()

Using device: cuda
Epoch  1: TrainTime = 11.13s, TrainLoss = 1.6950, TrainAcc = 38.65% 
Epoch  2: TrainTime =  9.66s, TrainLoss = 1.3754, TrainAcc = 50.27% 
Epoch  3: TrainTime =  9.41s, TrainLoss = 1.1898, TrainAcc = 57.54% 
Epoch  4: TrainTime = 10.43s, TrainLoss = 1.0774, TrainAcc = 61.92% 
Epoch  5: TrainTime = 13.01s, TrainLoss = 0.9887, TrainAcc = 64.89% 
Epoch  6: TrainTime =  9.64s, TrainLoss = 0.9197, TrainAcc = 67.53% 
Epoch  7: TrainTime =  8.90s, TrainLoss = 0.8702, TrainAcc = 69.25% 
Epoch  8: TrainTime = 11.68s, TrainLoss = 0.8248, TrainAcc = 70.98% 
Epoch  9: TrainTime = 13.79s, TrainLoss = 0.7804, TrainAcc = 72.63% 
Epoch 10: TrainTime =  9.79s, TrainLoss = 0.7502, TrainAcc = 73.55% 
Epoch 11: TrainTime =  8.85s, TrainLoss = 0.7296, TrainAcc = 74.20% 
Epoch 12: TrainTime =  9.83s, TrainLoss = 0.6992, TrainAcc = 75.25% 
Epoch 13: TrainTime = 13.10s, TrainLoss = 0.6707, TrainAcc = 76.41% 
Epoch 14: TrainTime = 11.99s, TrainLoss = 0.6449, TrainAcc = 77.44% 
Epoch 15: Train

```bash
(base) root@VM-0-80-ubuntu:/workspace/cifar10# nvidia-smi 
Sun Apr  5 22:54:43 2026       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 525.105.17   Driver Version: 525.105.17   CUDA Version: 12.0     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  Tesla T4            On   | 00000000:00:09.0 Off |                    0 |
| N/A   64C    P0    72W /  70W |   1308MiB / 15360MiB |     59%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
                                                                               
+-----------------------------------------------------------------------------+
| Processes:                                                                  |
|  GPU   GI   CI        PID   Type   Process name                  GPU Memory |
|        ID   ID                                                   Usage      |
|=============================================================================|
+-----------------------------------------------------------------------------+
```